# Means Are Hard, Covariances Are Easier

06/01 showed that ten years of data pin down an expected return only to within about ±10 points. The
same ten years pin down the volatility to within about ±2 points, and a correlation of 0.8 to within
about ±0.07. This asymmetry shapes all of quantitative portfolio construction: risk can be measured,
expected returns mostly cannot. This notebook quantifies it, shows what it does to an optimized portfolio,
and shows where covariances become hard too: when there are many assets.

## 0. Math link

- [Sample variance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/04_sample_variance.ipynb):
  $\hat\sigma^2$ is unbiased with $(n-1)$ in the denominator; here we need its **variance**, not only its mean.
- [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb)
  and the [covariance matrix](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/08_covariance_matrix.ipynb).
- The standard error of a mean return from [06/01](./01_how_noisy_is_a_mean_return.ipynb); the tangency portfolio
  from [04/04](../04_portfolio_mathematics/04_efficient_frontier_tangency_sharpe.ipynb).

## 1. Introduction

A risk report states "volatility 12.3%, expected return 7.1%", both estimated from the same five years of monthly data.
The two numbers look equally precise. They are not: the volatility is known to within about ±2 points, the expected
return to within about ±10. Mean-variance optimization treats both as exact, and the errors in the expected returns
dominate the result.

After this notebook you can:

- compute standard errors of a volatility and a correlation estimate, and compare them with the standard error of a mean;
- explain why higher-frequency data helps for volatility and correlation but not for the mean;
- account for fat tails in the standard error of volatility;
- show that errors in expected returns hurt an optimized portfolio far more than errors in covariances;
- recognize when the covariance matrix itself becomes unreliable: many assets, few observations.

## 2. Theory

### Mean vs volatility

With $T$ years of data at $m$ observations per year, $n = mT$, annual mean $\mu$ and volatility $\sigma$:

$$
\operatorname{SE}(\hat\mu) = \frac{\sigma}{\sqrt T}, \qquad
\operatorname{SE}(\hat\sigma) \approx \frac{\sigma}{\sqrt{2n}} = \frac{\sigma}{\sqrt{2mT}}
$$

The second follows from $\operatorname{Var}(\hat\sigma^2) \approx 2\sigma^4/n$ for normal returns and the delta method with $g(v) = \sqrt v$,
$g'(v) = 1/(2\sqrt v)$:
$\operatorname{Var}(\hat\sigma) \approx \frac{1}{4\sigma^2}\cdot\frac{2\sigma^4}{n} = \frac{\sigma^2}{2n}$.

Relative errors make the contrast plain:

$$
\frac{\operatorname{SE}(\hat\mu)}{\mu} = \frac{\sigma}{\mu}\cdot\frac{1}{\sqrt T}, \qquad
\frac{\operatorname{SE}(\hat\sigma)}{\sigma} = \frac{1}{\sqrt{2mT}}
$$

For equities $\sigma/\mu \approx 2.5$, so the mean has a relative error of $2.5/\sqrt T$; the volatility has $1/\sqrt{2mT}$, which also
shrinks with the sampling frequency $m$.

### Fat tails

For returns with kurtosis $\kappa$, $\operatorname{Var}(\hat\sigma^2) \approx (\kappa - 1)\sigma^4/n$ (06/02, exercise 6), so

$$
\operatorname{SE}(\hat\sigma) \approx \sigma\sqrt{\frac{\kappa - 1}{4n}}
$$

which reduces to $\sigma/\sqrt{2n}$ for $\kappa = 3$. With $\kappa = 6$ the standard error grows by $\sqrt{5/2} = 1.58$.

### Correlation

For bivariate normal returns, the sample correlation has (Fisher)

$$
\operatorname{SE}(\hat\rho) \approx \frac{1 - \rho^2}{\sqrt n}
$$

High correlations are estimated more precisely than low ones. Like volatility, the precision depends on $n$, so
higher-frequency data helps, as long as prices are recorded at the same time (non-synchronous closing prices across time
zones bias daily correlations toward zero).

### Why it matters for optimization

The tangency weights are proportional to $\Sigma^{-1}(\mu - r_f)$ (04/04). Small changes in $\mu$ move them a lot, because
$\Sigma^{-1}$ amplifies differences between correlated assets. Chopra and Ziemba (1993) estimate that errors in means are about
ten times as costly as errors in variances, and errors in variances about twice as costly as errors in covariances.

### Where covariances become hard

A covariance matrix of $N$ assets has $N(N+1)/2$ parameters. With $n$ observations and $N$ close to $n$ the sample covariance matrix
is nearly singular: its smallest eigenvalues are far too small and its largest far too large, even if every entry is
individually unbiased. For $N > n$ it is singular. Optimizers exploit the too-small eigenvalues, and the resulting
portfolios look much less risky in sample than they are (08/02 shrinkage addresses this).

## 3. Worked Examples

Swiss equities: $\mu = 6.5\%$, $\sigma = 16\%$; ten years of monthly data, $n = 120$.

### Example 1: mean vs volatility

$\operatorname{SE}(\hat\mu) = 16\%/\sqrt{10} = 5.06\%$, which is 78% of the 6.5% being estimated.

$\operatorname{SE}(\hat\sigma) = 16\%/\sqrt{240} = 1.03\%$, which is 6.5% of the 16% being estimated. The volatility is about twelve times more
precise in relative terms.

### Example 2: daily data

With 2,520 daily returns: $\operatorname{SE}(\hat\sigma) = 16\%/\sqrt{5040} = 0.23\%$; $\operatorname{SE}(\hat\mu)$ stays at 5.06%.

### Example 3: correlations

$\rho = 0.8$ (Swiss and global equities), $n = 120$: $\operatorname{SE} = (1 - 0.64)/\sqrt{120} = 0.033$.
$\rho = 0.1$ (Swiss equities and bonds): $\operatorname{SE} = 0.99/\sqrt{120} = 0.090$.

### Example 4: fat tails

Monthly kurtosis 6: $\operatorname{SE}(\hat\sigma) = 16\%\sqrt{5/480} = 1.63\%$ instead of 1.03%.

### Example 5: years for a 1-point standard error

Mean: $T = 16^2 = 256$ years. Volatility, monthly data: $2 \cdot 12 \cdot T = 256$, so $T = 10.7$ years. Volatility, daily data: $T = 0.5$ years.

## 4. Finance Application

### Where it matters

- **Risk models vs return forecasts.** Risk models (covariance matrices) are estimated from history with daily or
  weekly data and are fairly reliable over short horizons. Expected returns come from economic reasoning, analyst
  views or equilibrium arguments (CAPM, Black–Litterman in 08/04), not from historical averages.
- **Minimum-variance and risk-parity portfolios** use only the covariance matrix. Part of their popularity is that
  they avoid the noisiest input (08/03).
- **Mean-variance optimization** with historical means produces extreme, unstable portfolios (08/01). The
  problem is the means, not the method.
- **Large universes.** With 500 stocks and five years of monthly data ($n = 60$), the sample covariance matrix is
  singular. Factor models (07/03, 07/04) and shrinkage (08/02) impose structure.

### The usual misreadings

- "Both inputs come from the same data, so they are equally reliable." The volatility estimate is roughly ten times
  more precise in relative terms.
- "Use more frequent data to estimate everything better." Only second moments improve (06/01).
- "Each covariance entry is unbiased, so the matrix is fine." The matrix as a whole can be badly conditioned; what
  matters for optimization is its eigenvalues.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
months_per_year = 12
trading_days = 252

assets = ["Swiss equities", "global equities", "Swiss bonds"]
mu = np.array([0.065, 0.07, 0.02])                    # annual expected returns, decimal
vols = np.array([0.16, 0.17, 0.05])                   # annual volatilities
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma = np.outer(vols, vols) * corr                   # annual covariance matrix
rf = 0.005

### Estimation error of means, volatilities and correlations

5,000 histories of 10 years of monthly returns, multivariate normal.

In [ ]:
def simulate(n_paths, n_obs, periods_per_year, rng, df=None):
    # returns of shape (n_paths, n_obs, 3); df=None for normal, else multivariate t scaled to the same covariance
    L = np.linalg.cholesky(Sigma / periods_per_year)
    z = rng.standard_normal((n_paths, n_obs, 3))
    if df is not None:
        w = rng.chisquare(df, size=(n_paths, n_obs, 1)) / df
        z = z / np.sqrt(w) / np.sqrt(df / (df - 2))
    return mu / periods_per_year + z @ L.T


def estimates(R, periods_per_year):
    # annualized means, volatilities and the three pairwise correlations, per history
    m = R.mean(axis=1) * periods_per_year
    s = R.std(axis=1, ddof=1) * np.sqrt(periods_per_year)
    X = R - R.mean(axis=1, keepdims=True)
    C = np.einsum("pti,ptj->pij", X, X) / (R.shape[1] - 1)
    d = np.sqrt(np.einsum("pii->pi", C))
    rho = C / (d[:, :, None] * d[:, None, :])
    return m, s, rho[:, [0, 0, 1], [1, 2, 2]]


n_paths, years = 5_000, 10
R_m = simulate(n_paths, years * 12, 12, rng)
m_hat, s_hat, rho_hat = estimates(R_m, 12)

pairs = ["Swiss eq / global eq", "Swiss eq / bonds", "global eq / bonds"]
rho_true = corr[[0, 0, 1], [1, 2, 2]]
rows = []
for i, a in enumerate(assets):
    rows.append({"parameter": f"mean, {a}", "true": mu[i], "SE (simulated)": m_hat[:, i].std(),
                 "SE (formula)": vols[i] / np.sqrt(years)})
for i, a in enumerate(assets):
    rows.append({"parameter": f"vol, {a}", "true": vols[i], "SE (simulated)": s_hat[:, i].std(),
                 "SE (formula)": vols[i] / np.sqrt(2 * years * 12)})
for k, pname in enumerate(pairs):
    rows.append({"parameter": f"corr, {pname}", "true": rho_true[k], "SE (simulated)": rho_hat[:, k].std(),
                 "SE (formula)": (1 - rho_true[k] ** 2) / np.sqrt(years * 12)})
err = pd.DataFrame(rows).set_index("parameter")
err["SE / true"] = err["SE (simulated)"] / err["true"].abs().where(err["true"] != 0)
err.round(4)

The formulas match the simulation. Relative to the value being estimated, every mean has an error of 77–78%; every
volatility 6–7%; the 0.8 correlation 4%. The bond/global-equity correlation is zero, so a relative error is undefined
there; its standard error is 0.09.

### Daily data and fat tails

In [ ]:
R_d = simulate(1_000, years * trading_days, trading_days, rng)
m_d, s_d, rho_d = estimates(R_d, trading_days)
R_t = simulate(n_paths, years * 12, 12, rng, df=6)
m_t, s_t, rho_t = estimates(R_t, 12)
kurt_t6 = 3 + 6 / (6 - 4)

pd.DataFrame({
    "monthly, normal": [m_hat[:, 0].std(), s_hat[:, 0].std(), rho_hat[:, 0].std()],
    "daily, normal": [m_d[:, 0].std(), s_d[:, 0].std(), rho_d[:, 0].std()],
    "monthly, t(6)": [m_t[:, 0].std(), s_t[:, 0].std(), rho_t[:, 0].std()],
    "formula, monthly t(6)": [0.16 / np.sqrt(10), 0.16 * np.sqrt((kurt_t6 - 1) / (4 * 120)), np.nan],
}, index=["SE mean, Swiss eq", "SE vol, Swiss eq", "SE corr, Swiss/global eq"]).round(4)

Daily data shrinks the volatility and correlation errors by a factor of about $\sqrt{21} \approx 4.6$ and leaves the mean
untouched. Fat tails (kurtosis 6) widen the volatility error by about 1.5 (formula: 1.6), and the correlation error by
about 1.3.

### What the errors do to an optimized portfolio

For 2,000 of the histories, compute tangency-direction weights $w \propto \Sigma^{-1}(\mu - r_f)$, scaled to 10% volatility (the rest
in cash), in three ways: estimated $\mu$ with true $\Sigma$, true $\mu$ with estimated $\Sigma$, and both estimated. Evaluate each
portfolio with the **true** parameters.

In [ ]:
def scaled_tangency(mu_in, Sigma_in, target_vol=0.10):
    w = np.linalg.solve(Sigma_in, mu_in - rf)
    return w * target_vol / np.sqrt(w @ Sigma_in @ w)

def true_sharpe(w):
    return (w @ (mu - rf)) / np.sqrt(w @ Sigma @ w)

sr_max = np.sqrt((mu - rf) @ np.linalg.solve(Sigma, mu - rf))
w_opt = scaled_tangency(mu, Sigma)

n_opt = 2_000
W = {"estimated mu": [], "estimated Sigma": [], "both estimated": []}
for p in range(n_opt):
    S_hat = np.cov(R_m[p], rowvar=False) * 12
    W["estimated mu"].append(scaled_tangency(m_hat[p], Sigma))
    W["estimated Sigma"].append(scaled_tangency(mu, S_hat))
    W["both estimated"].append(scaled_tangency(m_hat[p], S_hat))
W = {k: np.array(v) for k, v in W.items()}

rows = []
for k, w in W.items():
    sr = np.array([true_sharpe(x) for x in w])
    rows.append({"inputs": k, "avg true SR": sr.mean(), "share of max SR": sr.mean() / sr_max,
                 "SR < 0": np.mean(sr < 0), **{f"std w {a}": w[:, i].std() for i, a in enumerate(assets)}})
opt_table = pd.DataFrame(rows).set_index("inputs")
print(f"true optimal weights at 10% vol: {dict(zip(assets, w_opt.round(3)))},  max SR {sr_max:.3f}")
opt_table.round(3)

With the true covariance matrix and estimated means, the portfolio reaches on average only 67% of the maximum Sharpe ratio of
0.49, and in 5.6% of histories its true Sharpe ratio is negative. With true means and an estimated covariance matrix it
reaches 99%. The weights tell the same story: their standard deviations are about five times larger when the means are
estimated. (The optimum holds 118% in bonds at 10% volatility: bonds are levered, financed by borrowing at the risk-free rate.)

### Many assets, few observations

Fifty assets, all with the same volatility (20% per year) and zero correlation, so the true covariance matrix has fifty
equal eigenvalues. Estimate it from 60 or 600 monthly returns and look at the eigenvalues; then build the minimum-variance
portfolio from the estimate and compare its in-sample volatility with its true volatility.

In [ ]:
N, vol_each = 50, 0.20
Sigma_N = np.eye(N) * vol_each**2

def eig_and_gmv(n_obs, rng):
    X = vol_each / np.sqrt(12) * rng.standard_normal((n_obs, N))
    S = np.cov(X, rowvar=False) * 12
    eig = np.sort(np.linalg.eigvalsh(S))[::-1]
    w = np.linalg.solve(S, np.ones(N))
    w /= w.sum()
    return eig, np.sqrt(w @ S @ w), np.sqrt(w @ Sigma_N @ w)

eig_60, in_60, true_60 = eig_and_gmv(60, rng)
eig_600, in_600, true_600 = eig_and_gmv(600, rng)
true_gmv = vol_each / np.sqrt(N)

print(f"true eigenvalues: all {vol_each**2:.3f}")
print(f"n = 60 : eigenvalues from {eig_60.min():.4f} to {eig_60.max():.3f}")
print(f"n = 600: eigenvalues from {eig_600.min():.4f} to {eig_600.max():.3f}")
print(f"\nminimum-variance portfolio, true optimum {true_gmv:.2%}")
print(f"n = 60 : in-sample vol {in_60:.2%}, true vol {true_60:.2%}")
print(f"n = 600: in-sample vol {in_600:.2%}, true vol {true_600:.2%}")

With 60 observations the sample eigenvalues range from 0.0005 to 0.146 around a true value of 0.04. The minimum-variance
portfolio built on that estimate reports 1.7% volatility in sample; its true volatility is 6.8%, more than twice the true
optimum of 2.8%. With 600 observations the spread narrows and the gap almost disappears.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = ["mean", "volatility"]
x_pos = np.arange(3)
rel_mean = err["SE (simulated)"].iloc[0:3].values / mu
rel_vol = err["SE (simulated)"].iloc[3:6].values / vols
ax.bar(x_pos - 0.2, rel_mean, width=0.4, label="mean")
ax.bar(x_pos + 0.2, rel_vol, width=0.4, label="volatility")
for i in range(3):
    ax.text(x_pos[i] - 0.2, rel_mean[i] + 0.01, f"{rel_mean[i]:.0%}", ha="center", fontsize=8)
    ax.text(x_pos[i] + 0.2, rel_vol[i] + 0.01, f"{rel_vol[i]:.0%}", ha="center", fontsize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(assets)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Standard error relative to the true value, 10 years of monthly data")
ax.set_ylabel("SE / true value")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.show()

In [ ]:
T_grid = np.linspace(1, 50, 200)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_grid, 0.16 / np.sqrt(T_grid), color="tab:red", label="mean (any frequency)")
ax.plot(T_grid, 0.16 / np.sqrt(2 * 12 * T_grid), color="tab:blue", label="volatility, monthly data")
ax.plot(T_grid, 0.16 / np.sqrt(2 * 252 * T_grid), color="tab:blue", linestyle="--", label="volatility, daily data")
ax.set_yscale("log")
ax.axhline(0.01, color="grey", linewidth=0.8, linestyle=":")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_title("Standard error of annual mean and volatility, Swiss equities (16% vol)")
ax.set_xlabel("years of data")
ax.set_ylabel("standard error (log scale)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for k, c in [("estimated mu", "tab:red"), ("estimated Sigma", "tab:blue")]:
    ax.scatter(W[k][:500, 0], W[k][:500, 2], s=6, alpha=0.4, color=c, label=f"{k}")
ax.plot(w_opt[0], w_opt[2], "k*", markersize=14, label="true optimum")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 2.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Tangency weights at 10% vol from 10 years of data (500 histories)")
ax.set_xlabel("weight in Swiss equities")
ax.set_ylabel("weight in Swiss bonds")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
k = np.arange(1, N + 1)
ax.plot(k, eig_60, "o-", markersize=3, label="sample, n = 60 months")
ax.plot(k, eig_600, "o-", markersize=3, label="sample, n = 600 months")
ax.axhline(vol_each**2, color="black", linewidth=1, label="true (all equal)")
ax.set_title("Eigenvalues of a 50 × 50 covariance matrix: true vs estimated")
ax.set_xlabel("eigenvalue rank")
ax.set_ylabel("eigenvalue (annual variance)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. Global equities, 17% volatility, 5 years of monthly data. Compute the standard errors of the annual mean and the annual
   volatility, absolute and relative to a 7% mean and 17% volatility.
2. Compute the standard error of a correlation estimate of 0.5 from 60 monthly returns. Would you be surprised by a sample
   correlation of 0.3?
3. A risk report shows "volatility 12.3%, expected return 7.1%", both from five years of monthly data for a portfolio with
   true volatility about 12%. **Interpret** the precision of each number for the investment committee.

### Intermediate

4. Simulate 5,000 correlation estimates from $n = 120$ bivariate normal observations for $\rho = 0.8$, $0.5$ and $0.1$. Compare the
   standard deviation of the estimates with $(1 - \rho^2)/\sqrt n$.
5. Repeat the optimization experiment with 30 years of data instead of 10. By how much does the share of the maximum Sharpe
   ratio improve with estimated $\mu$, and with estimated $\Sigma$?

### Advanced

6. For $N = 50$ uncorrelated assets, plot the in-sample and true volatility of the estimated minimum-variance portfolio as
   the number of monthly observations goes from 60 to 1,200. Why is the in-sample volatility too low?
7. Repeat the optimization experiment with $\Sigma$ estimated from 10 years of **daily** data (and $\mu$ still from monthly data,
   which gives the same estimate). How much does it help?

## 8. Solutions / Checks

In [ ]:
# 1.
se_m, se_s = 0.17 / np.sqrt(5), 0.17 / np.sqrt(2 * 60)
print(f"1. SE mean {se_m:.2%} ({se_m / 0.07:.0%} of 7%), SE vol {se_s:.2%} ({se_s / 0.17:.1%} of 17%)")

# 2.
se_r = (1 - 0.25) / np.sqrt(60)
print(f"2. SE corr {se_r:.3f}; 0.3 is {(0.5 - 0.3) / se_r:.1f} SE below 0.5")

# 3.
print(f"3. SE vol {0.12 / np.sqrt(120):.2%}, SE mean {0.12 / np.sqrt(5):.2%}")

# 4.
for r in [0.8, 0.5, 0.1]:
    C2 = np.array([[1, r], [r, 1]])
    Z = rng.standard_normal((5_000, 120, 2)) @ np.linalg.cholesky(C2).T
    Zc = Z - Z.mean(axis=1, keepdims=True)
    est = (Zc[..., 0] * Zc[..., 1]).sum(1) / np.sqrt((Zc[..., 0] ** 2).sum(1) * (Zc[..., 1] ** 2).sum(1))
    print(f"4. rho {r}: simulated SE {est.std():.4f}, formula {(1 - r**2) / np.sqrt(120):.4f}")

In [ ]:
# 5. and 7.
def opt_experiment(R_mu, R_sig, ppy_sig, n_runs):
    out = {"estimated mu": [], "estimated Sigma": []}
    for p in range(n_runs):
        m_est = R_mu[p].mean(axis=0) * 12
        S_est = np.cov(R_sig[p], rowvar=False) * ppy_sig
        out["estimated mu"].append(true_sharpe(scaled_tangency(m_est, Sigma)))
        out["estimated Sigma"].append(true_sharpe(scaled_tangency(mu, S_est)))
    return {k: np.mean(v) / sr_max for k, v in out.items()}

R30 = simulate(1_000, 360, 12, rng)
print("5. 30 years, monthly:", {k: round(v, 3) for k, v in opt_experiment(R30, R30, 12, 1_000).items()})
print("   10 years, monthly:", {k: round(v, 3) for k, v in opt_experiment(R_m, R_m, 12, 1_000).items()})
print("7. 10 years, Sigma from daily:", {k: round(v, 3) for k, v in opt_experiment(R_m, R_d, 252, 1_000).items()})

# 6.
n_grid = [60, 120, 240, 600, 1200]
res6 = [eig_and_gmv(n, rng)[1:] for n in n_grid]
for n, (ins, tru) in zip(n_grid, res6):
    print(f"6. n = {n:4d}: in-sample {ins:.2%}, true {tru:.2%}  (optimum {true_gmv:.2%})")

**1.** Mean: $17\%/\sqrt5 = 7.6\%$, 109% of the 7% being estimated. Volatility: $17\%/\sqrt{120} = 1.55\%$, 9.1% of 17%.

**2.** $(1 - 0.25)/\sqrt{60} = 0.097$. A sample value of 0.3 is about two standard errors below 0.5: unusual but not impossible.

**3.** The volatility estimate has a standard error of about 1.1 points, so 12.3% means "roughly 10–14.5%". The expected-return
estimate has a standard error of about 5.4 points, so 7.1% means "somewhere between about −4% and 18%". The committee
can use the volatility for risk budgeting; the expected return from five years of history is close to uninformative and
should be replaced or anchored by a forward-looking estimate.

**4.** The formula $(1-\rho^2)/\sqrt n$ matches the simulation closely for all three values; the estimate of 0.8 is almost three
times as precise as the estimate of 0.1.

**5.** With estimated $\mu$ the share rises from 68% to 86% of the maximum Sharpe ratio, so the portfolio still loses 14% after
30 years. With estimated $\Sigma$ it goes from 99.1% to 99.7%: it was already close.

**6.** In-sample volatility is too low because the optimizer puts weight on directions where the **sample** variance happens to
be small: it fits the noise. The true volatility of those weights is higher than the in-sample figure and higher than the
true optimum. The gap closes as $n$ grows relative to $N$.

**7.** Daily data brings the estimated-$\Sigma$ portfolio still closer to the optimum, but the estimated-$\mu$ portfolio is
unchanged: the mean cannot be improved by sampling more often.

## 9. Conclusion

- $\operatorname{SE}(\hat\mu) = \sigma/\sqrt T$ but $\operatorname{SE}(\hat\sigma) \approx \sigma/\sqrt{2n}$: with ten years of monthly data, an equity mean has a relative error of
  about 80%, a volatility about 6%.
- Volatility and correlation estimates improve with higher-frequency data; the mean does not. Fat tails widen the
  volatility error by $\sqrt{(\kappa-1)/2}$.
- $\operatorname{SE}(\hat\rho) \approx (1-\rho^2)/\sqrt n$: high correlations are estimated precisely.
- In an optimized portfolio, estimated means cost far more Sharpe ratio than an estimated covariance matrix.
- With many assets and few observations the covariance matrix becomes unreliable as a whole: extreme eigenvalues and
  too-optimistic in-sample risk.

Next: [Module 07, 01 Beta as a regression slope](../07_regression_and_factor_models/01_beta_as_a_regression_slope.ipynb).